**Vehicle Telemetry & Intelligent Fault Diagnosis System**
Developing a simulated vehicle telemetry pipeline that generates realistic vehicle operating data, injects configurable faults, and uses machine learning to identify abnormal behaviour and infer likely fault causes.

                    
                    
                    │      Driver      │
                    --------------------
                             |
                    ┌------------------┐
                    │ throttle/brake/  │
                    │ steering/gear    │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │       CAR        │
                    │                  │
                    │  ┌────────────┐  │
                    │  │   PARAMS   │  │
                    │  │ fixed      │  │
                    │  └────────────┘  │
                    │                  │
                    │  ┌────────────┐  │
                    │  │   STATE    │  │
                    │  │ changing   │  │
                    │  └────────────┘  │
                    │                  │
                    │ update_state()   │
                    └────────┬─────────┘
                             ↑
                    ┌────────┴─────────┐
                    │  FAULT INJECTOR  │
                    │ physical faults  │
                    └──────────────────┘
                             │
                             ↓
                    ┌──────────────────┐
                    │     SENSORS      │
                    │                  │
                    │ read(car.state)  │
                    │ + noise          │
                    │ + sensor faults  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │    TELEMETRY     │
                    │ timestamp +      │
                    │ sensor readings  │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │   ML DIAGNOSER   │
                    │                  │
                    │ fault detection  │
                    │ + classification │
                    └────────┬─────────┘
                             ↓
                    ┌──────────────────┐
                    │     OUTPUT       │
                    │ fault +          │
                    │ confidence       │
                    └──────────────────┘

In [1]:
# Purpose: imports — random for faults/maneuvers, numpy for pi/clamps
#Importing packages
import random as rd
import numpy as np

| Class / Component     | Main purpose                                     | Parameters / Data it holds | Main methods                                                                               |
| --------------------- | ------------------------------------------------ | -------------------------- | ------------------------------------------------------------------------------------------ |
| **`Car`**             | Represents the vehicle                           | `params`, `state`          | `update_state()`                                                                           |
| **`Physics`**         | Calculates how the car physically behaves        | No permanent data required | `update()`, `calculate_acceleration()`, `calculate_rpm()`, `calculate_engine_temp()`, etc. |
| **`FaultInjector`**   | Creates vehicle faults                           | `self.faults`              | `apply()`, `reset()`                                                                       |
| **`SensorSystem`**    | Simulates the vehicle's sensors                  | Sensor noise/settings      | `read()`, `add_noise()`                                                                    |
| **`TelemetrySystem`** | Converts sensor readings into recorded telemetry | Sampling/settings          | `process()`, `save()`                                                                      |
| **`MLDiagnoser`**     | Learns to identify the fault from telemetry      | Trained ML model           | `train()`, `predict()`, `evaluate()`                                                       |
| **`simulation()`**    | Runs the entire experiment                       | `fault`, `duration`, `dt`  | Runs the classes in sequence                                                               |
| **`dataset`**         | Stores results from every run                    | Global dictionary          | No methods                                                                                 |


In [2]:
# Purpose: Car — owns data (parameters fixed, state evolving). No math inside.
#Lets first create a blueprint of a Car and then establish the parameters for telemetry.
class Car():
    def __init__(self):
        #parameters() - SI units throughout
        self.parameters = {
            "engine": {"idle_rpm": 800, "max_rpm": 6500, "max_torque": 200, "final_drive": 3.5, "gear_ratios": [3.2, 2.1, 1.5, 1.1, 0.85, 0.7]},
            "vehicle": {"mass": 1500, "wheelbase": 2.7, "track": 1.6, "CdA": 0.7, "rho": 1.225, "g": 9.81},
            "cooling": {"efficiency": 1.0, "tau": 120, "ambient": 25, "heat_k": 0.08},
            "brakes": {"max_torque": 3000, "cool_tau": 80, "heat_k": 50.0},
            "tyres": {"radius": 0.32, "rr_base": 0.015, "tau": 60, "heat_k": 0.5}
        }
        self.state = {"speed": 0, "rpm": 800, "gear": 1, "engine_temp": 25, "brake_temp": 25, "wheel_speed": 0}

    
    def update_state(self, dt, thr, brk, steer, physics):
        speed, gear, rpm, engT, brkT, tyreT, ws, slip, yaw = physics.update(self, dt, thr, brk, steer)
        self.state["speed"] = speed
        self.state["gear"] = gear
        self.state["rpm"] = rpm
        self.state["engine_temp"] = engT
        self.state["brake_temp"] = brkT
        self.state["tyre_temp"] = tyreT
        self.state["wheel_speed"] = ws
        self.state["slip"] = slip
        self.state["yaw"] = yaw

        '''
        car = Car()
        physics = Physics()
        driver = Driver()
        thr, brk, steer = driver.inputs(t)
        '''


        
        



In [3]:
# Purpose: Physics — owns all math. Reads car, returns new values.
class Physics:

    def update(self, car, dt, thr, brk, steer):
       ''' pre-load:
           car = Car()
           physics = Physics()
           driver = Driver()
           thr, brk, steer = driver.inputs(t)
       '''

       F_traction, F_brake, a = self.calcaccln(car, thr, brk)
       speed, gear = self.calcspeed(car, a, dt)
       rpm = self.calcrpm(car)
       engine_temp = self.calcengtemp(car, thr, dt)
       brake_temp = self.calcbraketemp(car, brk, dt)
       tyre_temp = self.calctyretemp(car, dt)
       ws, slip, yaw = self.calcwheels(car, steer)



       return speed, gear, rpm, engine_temp, brake_temp, tyre_temp, ws, slip, yaw

    #do thr, brk, steer = driver.inputs(t)
    def calcaccln(self,car, thr, brk):
        engine_torque = car.parameters["engine"]["max_torque"]
        gear = car.state["gear"]  # 1-6(5 gears), initial 1
        speed = car.state["speed"] #needed for F-drag

        ratios = car.parameters["engine"]["gear_ratios"]
        ratio = ratios[gear - 1]
        final_drive = car.parameters["engine"]["final_drive"]

        Rraw = car.parameters["tyres"]["radius"]
        Rmean = sum(Rraw.values()) / 4 if isinstance(Rraw, dict) else Rraw

        Traw = car.parameters["brakes"]["max_torque"]
        if isinstance(Rraw, dict) or isinstance(Traw, dict):
            # per-wheel after fault — expand singles to dicts for uniform math
            if not isinstance(Traw, dict): Traw = {w: Traw for w in ["FL", "FR", "RL", "RR"]}
            if not isinstance(Rraw, dict): Rraw = {w: Rraw for w in ["FL", "FR", "RL", "RR"]}
            F_brake = brk * sum(Traw[w] / Rraw[w] for w in ["FL", "FR", "RL", "RR"])
        else:
            F_brake = brk * 4 * Traw / Rraw
        

        F_traction = thr * engine_torque * ratio * final_drive / Rmean

        m = car.parameters["vehicle"]["mass"]
        rho = car.parameters["vehicle"]["rho"]
        CdA = car.parameters["vehicle"]["CdA"]
        rr = car.parameters["tyres"]["rr_base"]
        g = car.parameters["vehicle"]["g"]

        F_drag = 0.5 * rho * CdA * speed ** 2
        F_rr = rr * m * g

        a = (F_traction - F_brake - F_drag - F_rr) / m
        return F_traction, F_brake, a

    #do F_traction, F_brake, a = physics.calcaccln()

    def calcspeed(self, car, a, dt):
        speed = max(0, car.state["speed"]+ a*dt ) 
        gear = car.state["gear"]
        rpm = car.state["rpm"]
        if speed < 2: gear = 1
        else:
            if rpm > 5500 and gear < 6: gear += 1
            elif rpm < 1500 and gear >1: gear -= 1
        return speed, gear
        # do speed, gear = calcspeed(car, a, dt)
    
    def calcrpm(self, car):
        speed = car.state["speed"]
        gear = car.state["gear"]
        final_drive = car.parameters["engine"]["final_drive"]
        radius = car.parameters["tyres"]["radius"]
        ratios = car.parameters["engine"]["gear_ratios"]
        ratio = ratios[gear - 1]
        Rmean = sum(radius.values()) / 4 if isinstance(radius, dict) else radius

        rpm = speed * ratio * final_drive * 60 / (2* np.pi* Rmean)
        return max(800, min(6500, rpm))
    
    def calcengtemp(self, car, thr, dt):
        #T += ((k1×thr×rpm - eff×(T-amb)) / tau)×dt
        cool = car.parameters["cooling"]
        eff = cool["efficiency"]
        tau = cool["tau"]
        amb = cool["ambient"]
        k = cool["heat_k"]

        Te = car.state["engine_temp"]
        rpm = car.state["rpm"]

        Te += ((k*thr*rpm - eff*(Te-amb)) / tau) * dt
        return max(amb, min(150, Te))
    
    def calcbraketemp(self, car, brk, dt):
        #T += ((heat_k×brk×speed - (T-amb)) / cool_tau)×dt
        cool = car.parameters["cooling"]
        amb = cool["ambient"]
        brake = car.parameters["brakes"]
        tau = brake["cool_tau"]
        k = brake["heat_k"]
        speed = car.state["speed"]
        Tb = car.state["brake_temp"]
        if not isinstance(Tb, dict): Tb = {w: Tb for w in ["FL", "FR", "RL", "RR"]}
        
        Traw = car.parameters["brakes"]["max_torque"]
        Tmap = Traw if isinstance(Traw, dict) else {w: Traw for w in ["FL", "FR", "RL", "RR"]}
        Tmean = sum(Tmap.values()) / 4

        out = {}
        for w in ["FL", "FR", "RL", "RR"]:
            heat = k * (Tmap[w] / Tmean) * brk * speed
            out[w] = max(amb, min(400, Tb[w] + ((heat - (Tb[w] - amb)) / tau) * dt))
        return out
    
    def calctyretemp(self, car, dt):
        amb = car.parameters["cooling"]["ambient"]
        tau = car.parameters["tyres"]["tau"]
        k = car.parameters["tyres"]["heat_k"]
        speed = car.state["speed"]
        slip = car.state.get("slip", {w: 0.02 for w in ["FL","FR","RL","RR"]})
        if not isinstance(slip, dict):
            slip = {w: 0.02 for w in ["FL","FR","RL","RR"]}
        Tt = car.state.get("tyre_temp", {w: amb for w in ["FL","FR","RL","RR"]})
        if not isinstance(Tt, dict):
            Tt = {w: Tt for w in ["FL","FR","RL","RR"]}

        out = {}
        for w in ["FL","FR","RL","RR"]:
            heat = k * slip[w] * speed
            out[w] = max(amb, min(120, Tt[w] + ((heat - (Tt[w]-amb)) / tau) * dt))
        return out

    def calcwheels(self, car, steer):
        import math
        speed = car.state["speed"]
        wb = car.parameters["vehicle"]["wheelbase"]
        track = car.parameters["vehicle"]["track"]
        Rraw = car.parameters["tyres"]["radius"]
        R = Rraw if isinstance(Rraw, dict) else {w: Rraw for w in ["FL","FR","RL","RR"]}

        yaw = speed * math.tan(steer) / wb if wb else 0.0  # 0 straight
        v = {}
        ws = {}
        for w in ["FL","FR","RL","RR"]:
            lateral = -yaw * track/2 if "L" in w else yaw * track/2
            v[w] = speed + lateral
            ws[w] = v[w] / R[w] if R[w] else 0.0
        slip = {w: (ws[w]*R[w]-speed)/max(speed,2) for w in ws}
        return ws, slip, yaw
        

        










        

In [4]:
# Purpose: FaultInjector — choose once per run, alter params once at onset. Hidden from ML.
class initFault:
    #iinitalise a fault
    def __init__(self):
        self.faults = [None, "cooling_failure","brake_failure","engine_failure","tyre_degradation"]
        self.location = ["FR","FL","RR","RL"]
        self.active = None

    def choose(self):
        fault = rd.choice(self.faults)
        severity = round(rd.uniform(0.1,1.0), 2)

        label = "mild" if severity <= 0.3 else "moderate" if severity <= 0.7 else "severe"

        onset = round(rd.uniform(20, 40), 1)  

        if fault in ("brake_failure", "tyre_degradation"):
            loc = rd.choice(self.location)
        else:
            loc = None


        self.active = {
            "fault": fault, "severity": severity,
            "location": loc, "onset": onset, 
            "label": label
        }

        return self.active

    def apply(self, car):
        if self.active is None or self.active["fault"] is None:
            return
        f, s, loc = self.active["fault"], self.active["severity"], self.active["location"]

        if f == "cooling_failure":
            car.parameters["cooling"]["efficiency"] = max(0.1, 1.0 - 0.85 * s)
        elif f == "brake_failure":
            
            # needs per-wheel dict — convert once if still single value
            if not isinstance(car.parameters["brakes"].get("max_torque"), dict):
                base = car.parameters["brakes"]["max_torque"]
                car.parameters["brakes"]["max_torque"] = {w: base for w in ["FL","FR","RL","RR"]}
            car.parameters["brakes"]["max_torque"][loc] *= (1.0 - 0.85 * s)
        elif f == "engine_failure":
            car.parameters["engine"]["max_torque"] *= (1.0 - 0.5 * s)
        elif f == "tyre_degradation":
            if not isinstance(car.parameters["tyres"].get("radius"), dict):
                base_r = car.parameters["tyres"]["radius"]
                car.parameters["tyres"]["radius"] = {w: base_r for w in ["FL","FR","RL","RR"]}
                car.parameters["tyres"]["rr"] = {w: 0.015 for w in ["FL","FR","RL","RR"]}
            car.parameters["tyres"]["radius"][loc] *= (1.0 - 0.05 * s)
            car.parameters["tyres"]["rr"][loc] = 0.015 + 0.02 * s

In [5]:
# Purpose: Driver — input generator. Maneuver -> thr/brk/steer per timestep.
class Driver:
    def __init__(self, maneuver):
        self.maneuver = maneuver
        # per-run jitter, fixed for whole run
        self.thr_jit = rd.uniform(-0.05, 0.05)
        self.t_brake_start = rd.uniform(28, 35)  # for brake_stop/mixed

    def inputs(self, t): #t is current clock 0 --> duration
        thr = brk = 0.0
        steer = 0.0
        j = self.thr_jit

        if self.maneuver == "cruise_low":
            thr = 0.25 + j
        elif self.maneuver == "cruise_high":
            thr = 0.50 + j
        elif self.maneuver == "accel":
            thr = 0.2 + (0.6 * min(t / 20.0, 1.0)) + j
        elif self.maneuver == "brake_stop":
            thr = 0.4 + j
            if self.t_brake_start <= t <= self.t_brake_start + 5:
                thr, brk = 0.0, 0.8
        elif self.maneuver == "mixed":  # cruise-accel-cruise-brake
            if t < 20: thr = 0.25 + j
            elif t < 40: thr = 0.2 + 0.6 * ((t-20)/20.0) + j
            elif t < 60: thr = 0.4 + j
            elif t < 65: thr, brk = 0.0, 0.8
            else: thr = 0.4 + j

        return max(0,min(1,thr)), max(0,min(1,brk)), steer

In [6]:
# Purpose: Telemetry — flat_row (dicts->columns) + run_once (one full run -> df+meta).
# new cell — simulation
def flat_row(t, thr, brk, steer, state):
    r = {"t": t, "thr": thr, "brk": brk, "steer": steer,
         "speed": state["speed"], "rpm": state["rpm"], "gear": state["gear"],
         "engine_temp": state["engine_temp"], "yaw": state.get("yaw", 0.0)}
    for w in ["FL","FR","RL","RR"]:
        bw = state["brake_temp"]
        tw = state.get("tyre_temp", {})
        ws = state["wheel_speed"]
        sl = state.get("slip", {})
        r[f"brake_{w}"] = bw[w] if isinstance(bw,dict) else bw
        r[f"tyre_{w}"] = tw[w] if isinstance(tw,dict) else tw
        r[f"ws_{w}"] = ws[w] if isinstance(ws,dict) else ws
        r[f"slip_{w}"] = sl[w] if isinstance(sl,dict) else sl
    return r

    
def run_once(maneuver=None, duration=120.0, dt=0.02):
    car = Car()
    physics = Physics()
    inj = initFault()
    sens = SensorSystem()
    active = inj.choose()  # fault, severity, location, onset, label

    if maneuver is None:
        maneuver = rd.choice(["cruise_low","cruise_high","accel","brake_stop","mixed"])
    driver = Driver(maneuver)

    applied = False
    rows = []
    t = 0.0
    while t <= duration:
        if not applied and t >= active["onset"]:
            inj.apply(car)
            applied = True
        thr, brk, steer = driver.inputs(t)
        car.update_state(dt, thr, brk, steer, physics)
        rows.append(sens.read(flat_row(t, thr, brk, steer, car.state)))
                    
        t += dt

    import pandas as pd
    df = pd.DataFrame(rows)
    return df, {"maneuver": maneuver, **active}

In [7]:
# Purpose: validation — forced fault (not random). lock_jit=True removes throttle noise; init_speed>0 gives rolling start for brake tests.
def run_forced(fault, location=None, severity=0.6, maneuver="cruise_low", duration=90, dt=0.05, lock_jit=True, init_speed=0):
    car, physics, inj = Car(), Physics(), initFault()
    inj.choose()
    inj.active.update({"fault": fault, "severity": severity, "location": location, "label": "moderate" if fault else "healthy", "onset": 30.0})
    car.state["speed"] = init_speed
    driver = Driver(maneuver)
    if lock_jit: driver.thr_jit = 0.0
    applied, rows, t = False, [], 0.0
    while t <= duration:
        if not applied and t >= 30.0:
            inj.apply(car); applied = True
        thr, brk, steer = driver.inputs(t)
        car.update_state(dt, thr, brk, steer, physics)
        rows.append(flat_row(t, thr, brk, steer, car.state))
        t += dt
    import pandas as pd
    return pd.DataFrame(rows)


In [8]:
# Purpose: SensorSystem — adds realistic sensor noise AFTER physics. ML trains on noisy telemetry, never clean state.
class SensorSystem:
    def __init__(self, speed_n=0.15, rpm_n=25, temp_n=0.5, ws_n=0.05):
        self.speed_n, self.rpm_n, self.temp_n, self.ws_n = speed_n, rpm_n, temp_n, ws_n
    def read(self, row):
        import numpy as np
        r = dict(row)
        r["speed"] += float(np.random.normal(0, self.speed_n))
        r["rpm"] += float(np.random.normal(0, self.rpm_n))
        r["engine_temp"] += float(np.random.normal(0, self.temp_n))
        for w in ["FL","FR","RL","RR"]:
            r[f"brake_{w}"] += float(np.random.normal(0, self.temp_n))
            r[f"tyre_{w}"] += float(np.random.normal(0, self.temp_n))
            r[f"ws_{w}"] += float(np.random.normal(0, self.ws_n))
        return r


In [9]:
# Purpose: validation gate — NORMAL vs faulty must diverge by eye before bulk dataset. Fails = fix heat_k, do not scale.
df_n = run_forced(None, None, 0.0, "cruise_low", 120, lock_jit=True)
df_f = run_forced("cooling_failure", None, 0.7, "cruise_low", 120, lock_jit=True)
print("NORMAL eng end:", round(float(df_n.engine_temp.iloc[-1]),1), "FAULT eng end:", round(float(df_f.engine_temp.iloc[-1]),1))
df_nb = run_forced(None, None, 0.0, "brake_stop", 120, lock_jit=True, init_speed=20)
df_fb = run_forced("brake_failure", "RR", 0.7, "brake_stop", 120, lock_jit=True, init_speed=20)
print("NORMAL brake_RR max:", round(float(df_nb.brake_RR.max()),1), "FAULT brake_RR max:", round(float(df_fb.brake_RR.max()),1))


NORMAL eng end: 80.6 FAULT eng end: 94.2
NORMAL brake_RR max: 40.4 FAULT brake_RR max: 33.8


In [10]:
# Purpose: Phase 1 dataset — n runs -> parquet + manifest.csv.
def build_dataset(n=5, duration=30.0, dt=0.05, outdir="data"):
    import os, pandas as pd
    os.makedirs(outdir, exist_ok=True)
    manifest = []
    for i in range(n):
        df, meta = run_once(None, duration, dt)  # random maneuver+fault
        df.to_parquet(f"{outdir}/run_{i:03d}.parquet", index=False)
        manifest.append({"run_id": f"{i:03d}", **meta})
    pd.DataFrame(manifest).to_csv(f"{outdir}/manifest.csv", index=False)
    return manifest

In [11]:
# Purpose: dataset switch — all Phase 2/3 table builders read DATADIR. data=50-run pilot, data200=stratified bulk.
DATADIR = "data200"


In [12]:
# Purpose: Phase 2 windowing — parquet time-series -> ML samples. 5s windows / 2.5s stride @20Hz. Pre-onset windows = NORMAL even in faulty runs.
import numpy as np
def make_windows(run_id, df, meta, win_s=5.0, stride_s=2.5, hz=20):
    win, stride = int(win_s*hz), int(stride_s*hz)
    rows = []
    for s in range(0, len(df)-win+1, stride):
        w = df.iloc[s:s+win]
        t_end = float(w["t"].iloc[-1])
        fault = meta.get("fault")
        import math as _m
        if fault is None or (isinstance(fault, float) and _m.isnan(fault)): fault = None
        onset = float(meta.get("onset", 1e9))
        label = "NORMAL" if (fault is None or t_end < onset) else fault
        f = {"run_id": run_id, "t_start": float(w["t"].iloc[0]), "t_end": t_end, "label": label,
             "location": (lambda v: None if (v is None or (isinstance(v, float) and v != v)) else v)(meta.get("location")), "severity": meta.get("severity"), "maneuver": meta.get("maneuver")}
        for col in ["speed","rpm","engine_temp","thr","brk"]:
            f[f"{col}_mean"] = float(w[col].mean()); f[f"{col}_std"] = float(w[col].std()); f[f"{col}_max"] = float(w[col].max())
        f["eng_dT"] = float(w["engine_temp"].iloc[-1] - w["engine_temp"].iloc[0]) / win_s

        long_i = max(0, s + win - 1 - int(30 * hz))
        span = max((s + win - 1 - long_i) / hz, 5.0)
        f["eng_dT_long"] = float(df["engine_temp"].iloc[s + win - 1] - df["engine_temp"].iloc[long_i]) / span
        m15_i = max(0, s + win - 1 - int(15 * hz))
        f["eng_mean_15"] = float(df["engine_temp"].iloc[m15_i:s + win].mean())
        
        for wname in ["FL","FR","RL","RR"]:
            f[f"ws_{wname}_mean"] = float(w[f"ws_{wname}"].mean())
        f["ws_LR"] = (f["ws_FL_mean"]+f["ws_RL_mean"])/2 - (f["ws_FR_mean"]+f["ws_RR_mean"])/2
        f["ws_FR_diff"] = f["ws_FL_mean"] - f["ws_FR_mean"]
        for wname in ["FL", "FR", "RL", "RR"]:
            f[f"brake_{wname}_mean"] = float(w[f"brake_{wname}"].mean())
            f[f"brake_{wname}_max"] = float(w[f"brake_{wname}"].max())
            f[f"tyre_{wname}_mean"] = float(w[f"tyre_{wname}"].mean())
            f[f"slip_{wname}_max"] = float(w[f"slip_{wname}"].abs().max())
        f["brk_temp_spread"] = max(f[f"brake_{w}_mean"] for w in ["FL","FR","RL","RR"]) - min(f[f"brake_{w}_mean"] for w in ["FL","FR","RL","RR"])
        f["slip_max"] = max(f[f"slip_{w}_max"] for w in ["FL","FR","RL","RR"])
        gear_mode = float(w["gear"].mode().iloc[0])
        ratio = [3.2, 2.1, 1.5, 1.1, 0.85, 0.7][int(gear_mode) - 1]
        rpm_exp = float(w["speed"].mean()) * ratio * 3.5 * 60 / (2 * np.pi * 0.32)
        f["rpm_resid"] = float(w["rpm"].mean()) - rpm_exp
        _gap5 = w["speed"].values * ratio * 3.5 * 60 / (2 * np.pi * 0.32) - w["rpm"].values
        f["sag_persist"] = float((_gap5 > 200).mean())  # 4. engine family: persistence over 5s window
        f["deficit_int"] = float(np.maximum(_gap5, 0).mean() / 1000.0)  # 4. engine family: accumulated shortfall over 5s
        f["torque_proxy"] = float(_gap5.mean() / 1000.0)  # 4. engine family: mean lean over 5s
        _b0 = max(0, s + win - 1 - 600)
        _bh = df.iloc[_b0:s + win]
        _gr = int(_bh["gear"].mode().iloc[0])
        _ra = [3.2, 2.1, 1.5, 1.1, 0.85, 0.7][_gr - 1]
        _gap = _bh["speed"].values * _ra * 3.5 * 60 / (2 * np.pi * 0.32) - _bh["rpm"].values
        f["sag30"] = float((_gap > 200).mean())  # 4. engine family: persistence over 30s lookback
        f["def30"] = float(np.maximum(_gap, 0).mean() / 1000.0)  # 4. engine family: accumulated shortfall (severity fuel)
        f["prox30"] = float(_gap.mean() / 1000.0)  # 4. engine family: mean lean, direction+depth
        rows.append(f)
    import pandas as pd
    return pd.DataFrame(rows)

# Demo: 1 NORMAL-ish + 1 faulty run -> window counts
import pandas as pd
m = pd.read_csv(DATADIR + "/manifest.csv")
demo = []
for _, r in m.head(4).iterrows():
    df = pd.read_parquet(f"{DATADIR}/run_{int(r['run_id']):03d}.parquet")
    demo.append(make_windows(r['run_id'], df, r.to_dict()))
import pandas as _pd
demo_df = _pd.concat(demo, ignore_index=True)
print(demo_df["label"].value_counts().to_string())
print("windows:", len(demo_df), "features:", len(demo_df.columns))


label
NORMAL    188
windows: 188 features: 56


In [13]:
# Purpose: Phase 2 anomaly — IsolationForest trained on NORMAL windows only. Flags ABNORMAL incl. unseen faults.
def build_window_table(n_runs=50):
    import pandas as pd
    m = pd.read_csv(DATADIR + "/manifest.csv").head(n_runs)
    parts = []
    for _, r in m.iterrows():
        df = pd.read_parquet(f"{DATADIR}/run_{int(r['run_id']):03d}.parquet")
        parts.append(make_windows(r['run_id'], df, r.to_dict()))
    import pandas as _pd
    return _pd.concat(parts, ignore_index=True)

def train_anomaly(df, feat_cols):
    from sklearn.ensemble import IsolationForest
    from sklearn.preprocessing import StandardScaler
    tr = df[df.label=="NORMAL"]
    sc = StandardScaler().fit(tr[feat_cols])
    cl = IsolationForest(n_estimators=200, contamination="auto", random_state=0).fit(sc.transform(tr[feat_cols]))
    return sc, cl

FEATS = [c for c in ["speed_mean","rpm_mean","engine_temp_mean","eng_dT","ws_LR","ws_FR_diff","thr_mean","brk_mean","brake_FL_mean", "brake_FR_mean", "brake_RL_mean", "brake_RR_mean", "brk_temp_spread", "tyre_FL_mean", "slip_max", "rpm_resid", "eng_dT_long", "eng_mean_15"] ]
W = build_window_table(12)
SC, CL = train_anomaly(W, FEATS)
import numpy as np
pred = CL.predict(SC.transform(W[FEATS]))  # 1 normal, -1 abnormal
W["pred_ab"] = (pred==-1).astype(int)
print("train windows:", len(W), "NORMAL:", int((W.label=="NORMAL").sum()))
print(W.groupby("label")["pred_ab"].mean().round(3).to_string())

train windows: 564 NORMAL: 564
label
NORMAL    0.137


In [14]:
# Purpose: Phase 2 fault classifier (cML) — RandomForest on ABNORMAL windows only. Group split by run (no leakage).
def build_full_table():
    import pandas as pd
    m = pd.read_csv(DATADIR + "/manifest.csv")
    parts = []
    for _, r in m.iterrows():
        df = pd.read_parquet(f"{DATADIR}/run_{int(r['run_id']):03d}.parquet")
        parts.append(make_windows(r['run_id'], df, r.to_dict()))
    import pandas as _pd
    return _pd.concat(parts, ignore_index=True)

FULL = build_full_table()
AB = FULL[FULL.label!="NORMAL"].copy()
# 1. proxy drop: remove driver-demand fingerprints (thr/brk/speed/steer) so the model diagnoses from vehicle response, not maneuver identity
PROXY = ("thr_","brk_","speed_","steer")
CFEATS = [c for c in FULL.columns if c not in ("run_id","label","location","severity","maneuver","t_start","t_end") and not c.startswith(PROXY)]  # drop driver-demand proxies: diagnose from response, not maneuver
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
gss = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=0).split(AB[CFEATS], AB["label"], AB["run_id"])
tri, tei = next(gss)
TRF = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(AB[CFEATS].iloc[tri], AB["label"].iloc[tri])
print("abnormal windows:", len(AB), "train:", len(tri), "test:", len(tei))
from sklearn.metrics import accuracy_score, confusion_matrix
pr = TRF.predict(AB[CFEATS].iloc[tei])
print("type accuracy:", round(accuracy_score(AB["label"].iloc[tei], pr), 3))
print("labels:", list(TRF.classes_))
print(confusion_matrix(AB["label"].iloc[tei], pr))


abnormal windows: 5806 train: 4341 test: 1465
type accuracy: 0.789
labels: ['brake_failure', 'cooling_failure', 'engine_failure', 'tyre_degradation']
[[265  37  13   7]
 [104 317  22   2]
 [ 51  68 214   2]
 [  1   0   2 360]]


In [15]:
# 3. calibration + abstention (canon step 3): confidence must track accuracy; report-if-confident policy.
P = TRF.predict_proba(AB[CFEATS].iloc[tei])
conf = P.max(1)
pred = TRF.classes_[P.argmax(1)]
true = AB["label"].iloc[tei].values
print("base acc:", round((pred == true).mean(), 3))
for lo in [0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    hi = lo + 0.1 if lo < 0.9 else 1.01
    m = (conf >= lo) & (conf < hi)
    if m.sum():
        print(f"{lo:.1f}-{hi:.2f}: acc={((pred[m] == true[m]).mean()):.3f} n={m.sum()}")
for t in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    m = conf >= t
    print(f"t={t}: coverage={m.mean():.3f} acc={((pred[m] == true[m]).mean() if m.sum() else float('nan')):.3f}")


base acc: 0.789
0.4-0.50: acc=0.651 n=278
0.5-0.60: acc=0.675 n=271
0.6-0.70: acc=0.765 n=213
0.7-0.80: acc=0.885 n=148
0.8-0.90: acc=0.952 n=124
0.9-1.01: acc=1.000 n=326
t=0.0: coverage=1.000 acc=0.789
t=0.5: coverage=0.739 acc=0.851
t=0.6: coverage=0.554 acc=0.910
t=0.7: coverage=0.408 acc=0.962
t=0.8: coverage=0.307 acc=0.987
t=0.9: coverage=0.223 acc=1.000


In [16]:
# Purpose: Phase 3 localisation — predict wheel (FL/FR/RL/RR) or system (cooling/engine). Trained on abnormal windows with true type as extra feat.
LOCT = AB.copy()
LOCT["loc"] = LOCT.apply(lambda r: r["location"] if r["label"] in ("brake_failure","tyre_degradation") else "system", axis=1)
LOCT["pred_type"] = TRF.predict(AB[CFEATS])
LOCD = __import__("pandas").get_dummies(LOCT["pred_type"], prefix="pt")
LOCF = CFEATS + list(LOCD.columns)
LOCT = __import__("pandas").concat([LOCT, LOCD], axis=1)
from sklearn.model_selection import GroupShuffleSplit as _G
from sklearn.ensemble import RandomForestClassifier as _R
g2 = _G(n_splits=1, test_size=0.25, random_state=1).split(LOCT[LOCF], LOCT["loc"], LOCT["run_id"])
a, b = next(g2)
LOC_COLS = list(LOCD.columns)
LRF = _R(n_estimators=300, min_samples_leaf=5, random_state=1, n_jobs=-1).fit(LOCT[LOCF].iloc[a], LOCT["loc"].iloc[a])
from sklearn.metrics import accuracy_score as _a
print("loc accuracy:", round(_a(LOCT["loc"].iloc[b], LRF.predict(LOCT[LOCF].iloc[b])), 3))
print("classes:", list(LRF.classes_))


loc accuracy: 0.73
classes: ['FL', 'FR', 'RL', 'RR', 'system']


In [17]:
# Purpose: Phase 3 severity — RandomForest regressor on abnormal windows -> numeric severity_s. Group split by run.
SEV = AB.copy()
from sklearn.model_selection import StratifiedGroupKFold as _S
from sklearn.ensemble import RandomForestRegressor as _R
import numpy as _np
SEV["bucket"] = _np.where(SEV["severity"]<=0.33, 0, _np.where(SEV["severity"]<=0.66, 1, 2))
gs = _S(n_splits=4, shuffle=True, random_state=2).split(SEV[CFEATS], SEV["bucket"], SEV["run_id"])
sa, sb = next(gs)
SRF = _R(n_estimators=300, min_samples_leaf=5, random_state=2, n_jobs=-1).fit(SEV[CFEATS].iloc[sa], SEV["severity"].iloc[sa])
from sklearn.metrics import mean_absolute_error as _m
pr = SRF.predict(SEV[CFEATS].iloc[sb])
print("severity MAE:", round(_m(SEV["severity"].iloc[sb], pr), 3))
import pandas as _p
_r = _p.DataFrame({"true": SEV["severity"].iloc[sb].values, "pred": pr, "fault": SEV["label"].iloc[sb].values})
print(_r.groupby("fault").apply(lambda g: round(_m(g["true"], g["pred"]), 3)).to_string())


severity MAE: 0.187
fault
brake_failure       0.244
cooling_failure     0.206
engine_failure      0.183
tyre_degradation    0.089


/var/folders/nx/30hzfndn1qv9997_rx8zt7680000gn/T/ipykernel_5664/3083331882.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(_r.groupby("fault").apply(lambda g: round(_m(g["true"], g["pred"]), 3)).to_string())


In [18]:
# Purpose: stratified bulk data (200 runs: 5 faults x 5 maneuvers x 8 each) -> data200/ + manifest. Severity/location/onset random inside. Takes ~10-15 min on i7.
def build_bulk_dataset(n_per_combo=8, duration=120.0, dt=0.05, outdir="data200"):
    import os, pandas as pd
    os.makedirs(outdir, exist_ok=True)
    faults = [None, "cooling_failure", "brake_failure", "engine_failure", "tyre_degradation"]
    maneuvers = ["cruise_low", "cruise_high", "accel", "brake_stop", "mixed"]
    manifest, i = [], 0
    for f in faults:
        for man in maneuvers:
            for _ in range(n_per_combo):
                car, physics, inj = Car(), Physics(), initFault()
                inj.choose()
                inj.active["fault"] = f
                from math import isnan as _n
                inj.active["location"] = __import__("random").choice(["FL","FR","RL","RR"]) if f in ("brake_failure","tyre_degradation") else None
                if f is None: inj.active["severity"], inj.active["label"] = 0.0, "healthy"
                driver = Driver(man)
                applied, rows, t = False, [], 0.0
                sens = SensorSystem()
                while t <= duration:
                    if not applied and t >= inj.active["onset"]:
                        inj.apply(car); applied = True
                    thr, brk, steer = driver.inputs(t)
                    car.update_state(dt, thr, brk, steer, physics)
                    rows.append(sens.read(flat_row(t, thr, brk, steer, car.state)))
                    t += dt
                df = pd.DataFrame(rows)
                df.to_parquet(f"{outdir}/run_{i:03d}.parquet", index=False)
                manifest.append({"run_id": f"{i:03d}", "maneuver": man, **inj.active})
                i += 1
    pd.DataFrame(manifest).to_csv(f"{outdir}/manifest.csv", index=False)
    return manifest

# smoke: 1 per combo = 25 quick short runs into bulkrun_smoke/
# smoke = build_bulk_dataset(1, 20, 0.05, "data_smoke")


In [19]:
# Purpose: generalisation evaluation — leave-one-fault-out (novelty) + unseen-maneuver. Caches windows table to windows_DATADIR.parquet.
import os as _os, pandas as _pd
WTAB = DATADIR + "/windows.parquet"
if _os.path.exists(WTAB):
    EV = _pd.read_parquet(WTAB)
else:
    _m = _pd.read_csv(DATADIR + "/manifest.csv")
    _p = []
    for _, _r in _m.iterrows():
        _d = _pd.read_parquet(f"{DATADIR}/run_{int(_r['run_id']):03d}.parquet")
        _p.append(make_windows(_r['run_id'], _d, _r.to_dict()))
    EV = _pd.concat(_p, ignore_index=True)
    EV.to_parquet(WTAB, index=False)
print("EV windows:", len(EV), "cols:", len(EV.columns))
assert all(k in EV.columns for k in ("sag_persist", "deficit_int", "torque_proxy", "sag30", "def30", "prox30")), "stale windows cache: delete data200/windows.parquet and re-run"

PROXY = ("thr_","brk_","speed_","steer")
CFEATS = [c for c in EV.columns if c not in ("run_id","label","location","severity","maneuver","t_start","t_end") and not c.startswith(PROXY)]
# A. leave-one-fault-out: anomaly must flag held-out fault as novel
from sklearn.ensemble import IsolationForest as _I
from sklearn.preprocessing import StandardScaler as _S
for _held in ["brake_failure", "cooling_failure", "engine_failure", "tyre_degradation"]:
    _tr = EV[(EV.label=="NORMAL") & (~EV.run_id.isin(EV[EV.label==_held].run_id.unique()))]
    _te = EV[EV.label==_held]
    _sc = _S().fit(_tr[CFEATS])
    _cl = _I(n_estimators=200, contamination="auto", random_state=0).fit(_sc.transform(_tr[CFEATS]))
    _flag = (_cl.predict(_sc.transform(_te[CFEATS]))==-1).mean()
    print(f"hold {_held}: novel-flag rate {round(float(_flag),3)} (n={len(_te)})")


EV windows: 9400 cols: 56
hold brake_failure: novel-flag rate 0.251 (n=1421)
hold cooling_failure: novel-flag rate 0.262 (n=1490)
hold engine_failure: novel-flag rate 0.207 (n=1447)
hold tyre_degradation: novel-flag rate 0.364 (n=1448)


In [24]:
# Purpose: Phase 4 diagnostic report — per-run evidence from features (thresholds, not LLM prose).
def diagnose_run(run_id):
    verdict = True
    import pandas as _p, collections as _c
    df = _p.read_parquet(f"{DATADIR}/run_{int(run_id):03d}.parquet")
    man = _p.read_csv(DATADIR + "/manifest.csv")
    meta = man[man.run_id==int(run_id)].iloc[0].to_dict()
    W = make_windows(run_id, df, meta)
    _fl = meta.get("fault")
    _on = float(meta.get("onset", 0))
    post = W[W.t_end>=_on] if _fl is not None and str(_fl)!="nan" else W
    _af = (CL.predict(SC.transform(W[FEATS]))==-1)
    status = "ABNORMAL" if _af.mean()>=0.08 else "NORMAL"  # calibrated: NORMAL≈0.06, cruise_low faults 0.09+; mild-cruise blind spot documented
    ab = TRF.predict(post[CFEATS])
    top, n = _c.Counter(ab).most_common(1)[0]
    import pandas as _q
    _pt = _q.get_dummies(_p.Series(TRF.predict(post[CFEATS])), prefix="pt").reindex(columns=LOC_COLS, fill_value=0)
    _pf = _p.concat([post[CFEATS].reset_index(drop=True), _pt.reset_index(drop=True)], axis=1)
    loctop = _c.Counter(LRF.predict(_pf[LOCF])).most_common(1)[0][0]
    sev = float(SRF.predict(post[CFEATS]).mean())
    ev = ["eng_dT_long max %.2f C/s" % W["eng_dT_long"].max(), "brake spread max %.1f C" % W["brk_temp_spread"].max(), "rpm_resid min %.0f rpm" % W["rpm_resid"].min(), "slip max %.3f" % W["slip_max"].max()]
    fault_ = top if status=='ABNORMAL' else "none"
    verdit = True if fault_ == _fl else False
    return {"run": int(run_id), "status": status, "fault": (top if status=="ABNORMAL" else "none"), "conf": round(n/len(ab),2), "location": loctop, "severity_s": round(sev,2), "evidence": ev, "truth": _fl}, verdict

print(diagnose_run(7))
print(diagnose_run(40))


({'run': 7, 'status': 'NORMAL', 'fault': 'none', 'conf': 0.6, 'location': 'FR', 'severity_s': 0.5, 'evidence': ['eng_dT_long max 0.47 C/s', 'brake spread max 0.2 C', 'rpm_resid min -922 rpm', 'slip max 0.000'], 'truth': nan}, True)
({'run': 40, 'status': 'ABNORMAL', 'fault': 'cooling_failure', 'conf': 1.0, 'location': 'system', 'severity_s': 0.52, 'evidence': ['eng_dT_long max 0.50 C/s', 'brake spread max 0.3 C', 'rpm_resid min -714 rpm', 'slip max 0.000'], 'truth': 'cooling_failure'}, True)


In [47]:
#random run spot-check (diagnose_run returns (report, ok) tuple)
import numpy as np
info, _ = diagnose_run(int(np.random.randint(0, 200)))
truth = info["truth"]
called = info["fault"] if info["status"]=="ABNORMAL" else None
ok = (str(truth)=="nan" and called is None) or (str(truth)!="nan" and called==truth)
print("CORRECT" if ok else "WRONG", info)
